In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

尝试用手动创建虚拟变量的方式，对双边固定+时间固定进行建模（内存只能支持1000个国家对，而实际上有27,497个）

In [2]:
def add_interaction_terms(df, spei_var='spei_lag1'):
    """
    只生成与 SPEI 的交互项（主效应在 Pair FE 下会被吸收，不进入模型）
    """
    if 'o_MA' in df.columns:
        df['spei_ma_interact'] = df[spei_var] * df['o_MA']
    else:
        df['spei_ma_interact'] = np.nan  # 若缺失该列，保持列存在性

    if 'border_friction_ij' in df.columns:
        df['spei_bf_interact'] = df[spei_var] * df['border_friction_ij']
    else:
        df['spei_bf_interact'] = np.nan

    return df

In [3]:
def build_year_month(df, date_col='migration_date', base_year=2019):
    dt = pd.to_datetime(df[date_col])
    df['year_month'] = (dt.dt.year - base_year) * 12 + dt.dt.month
    return df


In [4]:
def prepare_data_for_pair_time_fe(
    data,
    spei_var='spei_lag1',
    origin_col='origin_iso3',
    dest_col='destination_iso3',
    date_col='migration_date',
    max_pairs=None,
    min_obs_per_pair=2
):
    """
    - 生成 year_month、pair_id
    - 仅保留必要列并 dropna
    - 可选：保留出现次数最多的前 max_pairs 个 pair；剔除观测过少的 pair
    - 返回：data_clean
    """
    # 构造时间
    data = build_year_month(data.copy(), date_col=date_col)

    # 构造 pair_id（建议用原始字符串，不 factorize；便于聚类分组）
    data['pair_id'] = data[origin_col].astype(str) + '→' + data[dest_col].astype(str)

    # 交互项
    data = add_interaction_terms(data, spei_var=spei_var)

    # 选择变量
    base_vars = [
        'log_flow', spei_var, 'spei_ma_interact', 'spei_bf_interact',
        'year_month', 'pair_id',
        # 控制变量（时间变的保留；时间不变的 pair-level 会被 FE 吸收，主效应不进 X）
        'o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban',
        'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability',
        # 下列主效应会被吸收：仅用于生成交互；不进模型主效应
        'o_MA', 'border_friction_ij', 'control_distwces'
    ]
    available = [c for c in base_vars if c in data.columns]
    data_clean = data[available].dropna(subset=['log_flow', spei_var, 'year_month', 'pair_id']).copy()

    # 最小观测阈值（避免单例 pair 导致秩亏/极大方差）
    if min_obs_per_pair is not None and min_obs_per_pair > 0:
        counts = data_clean['pair_id'].value_counts()
        keep_pairs = counts[counts >= min_obs_per_pair].index
        before = len(data_clean)
        data_clean = data_clean[data_clean['pair_id'].isin(keep_pairs)]
        print(f"按 pair 最小样本阈值({min_obs_per_pair})过滤: {before:,} -> {len(data_clean):,}")

    # 仅限制 pair 数量（可选，控制内存）
    if max_pairs is not None:
        top_pairs = data_clean['pair_id'].value_counts().head(max_pairs).index
        before = len(data_clean)
        data_clean = data_clean[data_clean['pair_id'].isin(top_pairs)]
        print(f"限制前 {max_pairs} 个最常见 pair：{before:,} -> {len(data_clean):,}")
    else:
        print("未限制 pair 数量。")

    # 统计信息
    print("固定效应面板维度：")
    print(f"  Pair 数量: {data_clean['pair_id'].nunique():,}")
    print(f"  时间点数量: {data_clean['year_month'].nunique():,}")
    print(f"  观测数: {len(data_clean):,}")

    return data_clean

In [5]:
def create_pair_time_fe_dummies(df):
    """
    - Pair FE: 对 pair_id 建虚拟变量
    - Time FE: 对 year_month 建虚拟变量
    - drop_first=True + add_constant 防止虚拟变量陷阱
    """
    pair_dummies = pd.get_dummies(df['pair_id'], prefix='FE_pair', drop_first=True, dtype=int)
    time_dummies = pd.get_dummies(df['year_month'], prefix='FE_time', drop_first=True, dtype=int)
    print(f"FE 列数量：pair={pair_dummies.shape[1]}, time={time_dummies.shape[1]}, 合计={pair_dummies.shape[1]+time_dummies.shape[1]}")
    return pair_dummies, time_dummies


In [6]:
def fit_ols_with_cluster(y, X, cluster_groups):
    X = sm.add_constant(X, has_constant='add').astype(float)
    y = y.astype(float)
    # statsmodels 支持字符串分组；如需可转为 codes：pd.factorize(cluster_groups)[0]
    model = sm.OLS(y, X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': cluster_groups})
    return res

In [7]:
def build_X_for_model(df, pair_dummies, time_dummies, model_id, spei_var='spei_lag1'):
    """
    Model 设计（Pair + Time FE）：
      M1: spei + [可选的时间变控制] + FE_pair + FE_time
      M2: M1 + spei*o_MA（不含 o_MA 主效应）
      M3: M1 + spei*border_friction（不含 border_friction 主效应）
      M4: M1 + spei*o_MA + spei*border_friction（不含二者主效应）
    注意：control_distwces、border_friction_ij、o_MA 的主效应在 Pair FE 下被吸收，不进主效应。
    """
    # 时间变化的控制变量（可根据你的口径调整）
    time_varying_controls = [c for c in [
        'o_gdp_pc', 'd_gdp_pc', 'o_pop', 'd_pop', 'o_urban', 'd_urban',
        'o_unemp', 'd_unemp', 'o_political_stability', 'd_political_stability'
    ] if c in df.columns]

    cols = [spei_var] + time_varying_controls

    if model_id in (2, 4) and 'spei_ma_interact' in df.columns:
        cols.append('spei_ma_interact')   # 仅加交互，不加 o_MA 主效应

    if model_id in (3, 4) and 'spei_bf_interact' in df.columns:
        cols.append('spei_bf_interact')   # 仅加交互，不加 border_friction 主效应

    X_main = df[cols].copy()
    X = pd.concat([X_main, pair_dummies, time_dummies], axis=1)
    return X

In [8]:
def run_all_models_pair_time_fe(df, spei_var='spei_lag1'):
    """
    运行 Model 1–4，并返回结果字典
    """
    pair_dum, time_dum = create_pair_time_fe_dummies(df)
    y = df['log_flow']
    groups = df['pair_id']  # 用于聚类稳健标准误

    results = {}
    for mid in [1, 2, 3, 4]:
        X = build_X_for_model(df, pair_dum, time_dum, mid, spei_var=spei_var)
        res = fit_ols_with_cluster(y, X, groups)
        results[mid] = res
        print(f"✅ Model {mid} 完成：K={X.shape[1]} 列，N={int(res.nobs):,}")
    return results

In [9]:
def summarize_result(res, label='Model'):
    params = res.params
    bse = res.bse
    def fmt(name):
        if name in params.index:
            return f"{params[name]:.4g} ({bse[name]:.4g})"
        return "— (—)"

    keys = ['spei_lag1', 'spei_ma_interact', 'spei_bf_interact']
    line = ", ".join([f"{k}: {fmt(k)}" for k in keys])
    print("="*60)
    print(f"{label} 摘要")
    print("="*60)
    print(line)
    print(f"N={int(res.nobs):,} | R²={res.rsquared:.4f} | adj.R²={res.rsquared_adj:.4f}")
    print(f"F({res.df_model:.0f},{res.df_resid:.0f}) p={res.f_pvalue:.4g}")
    print("（按 pair 聚类的稳健标准误）\n")


In [14]:
data = pd.read_csv('../data/processed/merged_dataset_cleaned.csv')

# 准备（Pair + Time FE）
data = prepare_data_for_pair_time_fe(
    data,
    spei_var='spei_lag1',
    origin_col='origin_iso3',
    dest_col='destination_iso3',
    date_col='migration_date',
    max_pairs=None,          # 如内存吃紧，可设为比如 50_000 或 20_000
    min_obs_per_pair=2       # 建议至少 2，避免单例 pair
)

# 运行四个模型
results = run_all_models_pair_time_fe(data, spei_var='spei_lag1')

# 摘要展示
for k in [1,2,3,4]:
    summarize_result(results[k], label=f"Model {k}")

按 pair 最小样本阈值(2)过滤: 1,319,786 -> 1,319,786
未限制 pair 数量。
固定效应面板维度：
  Pair 数量: 27,497
  时间点数量: 48
  观测数: 1,319,786


MemoryError: Unable to allocate 270. GiB for an array with shape (1319786, 27497) and data type int64